<a href="https://colab.research.google.com/github/Syeda-Rb-47/Building-GlueScore--An-Interpretable-Glue-Likeness-Scoring-Pipeline/blob/main/5_Development_of_an_Interpretable_Glue_Likeness_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTS

In [ ]:
# Importing required libraries and mounting DRIVE.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import joblib
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Defining path of datasets.
DRIVE_DIR = "/content/drive/MyDrive/Thesis Datasets"

In [ ]:
# Loading the dataset.
df = pd.read_csv(DRIVE_DIR + "/final_descriptor_matrix.csv")

meta_cols = ["Master_ID", "Source_ID", "Label", "Canonical_SMILES", "StdInChIKey"]
meta_cols = [c for c in meta_cols if c in df.columns]
desc_cols = [c for c in df.columns if c not in meta_cols]

glues = df[df["Label"] == 1]
bg    = df[df["Label"] == 0]

print("Dataset Loaded\n")
print("Total molecules :", len(df))
print("Glues           :", len(glues))
print("Background      :", len(bg))
print("Final descriptor:", len(desc_cols))
print("Final descriptor:", desc_cols)

Dataset Loaded

Total molecules : 3738
Glues           : 1623
Background      : 2115
Final descriptor: 7
Final descriptor: ['fr_NH1', 'NumHeterocycles', 'NOCount', 'RingCount', 'NumAromaticRings', 'fr_aniline', 'FractionCSP3']


TRAIN/TEST SPLIT

In [ ]:
# Splitting the data into Train and Test sets (80/20, stratified).
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Label"],
    random_state=42
)

print("Train/Test Split (80/20, stratified)\n")
print("Train total :", len(train_df))
print("Glues (Train):", (train_df["Label"]==1).sum(), " , Background (Train):", (train_df["Label"]==0).sum())
print("Test total  :", len(test_df))
print("Glues (Test):", (test_df["Label"]==1).sum(),  " , Background (Test):", (test_df["Label"]==0).sum())

Train/Test Split (80/20, stratified)

Train total : 2990
Glues (Train): 1298  , Background (Train): 1692
Test total  : 748
Glues (Test): 325  , Background (Test): 423


DESCRIPTOR STANDARDIZATION

In [ ]:
# Standardizing Descriptors (fit on Train set only, then applied to both Train and Test sets).
scaler = StandardScaler()
scaler.fit(train_df[desc_cols].values)

z_train = pd.DataFrame(scaler.transform(train_df[desc_cols].values), columns=desc_cols, index=train_df.index)
z_train["Label"] = train_df["Label"].values

z_test = pd.DataFrame(scaler.transform(test_df[desc_cols].values), columns=desc_cols, index=test_df.index)
z_test["Label"] = test_df["Label"].values

z_glues = z_train[z_train["Label"] == 1][desc_cols]
z_bg    = z_train[z_train["Label"] == 0][desc_cols]

print("Standardization Complete (scaler fit on train only)")
print("\nMean standardized values of Glues (Train set):")
print(z_glues.mean().round(4))
print("\nMean standardized values of Background (Train set):")
print(z_bg.mean().round(4))

Standardization Complete (scaler fit on train only)

Mean standardized values of Glues (Train set):
fr_NH1              0.6125
NumHeterocycles     0.5829
NOCount             0.4759
RingCount           0.5233
NumAromaticRings    0.4748
fr_aniline          0.3675
FractionCSP3       -0.3098
dtype: float64

Mean standardized values of Background (Train set):
fr_NH1             -0.4699
NumHeterocycles    -0.4471
NOCount            -0.3651
RingCount          -0.4014
NumAromaticRings   -0.3643
fr_aniline         -0.2820
FractionCSP3        0.2376
dtype: float64


DELTA, WEIGHT AND SIGN CALCULATION

In [ ]:
# Calculating Delta, Weights and Signs (Train set only).
weight_results = []

for desc in desc_cols:
    delta     = z_glues[desc].mean() - z_bg[desc].mean()
    weight    = abs(delta)
    sign      = 1 if delta > 0 else -1
    direction = "Glues_Higher" if delta > 0 else "Glues_Lower"

    weight_results.append({
        "Descriptor" : desc,
        "Delta"      : round(delta, 4),
        "Weight"     : round(weight, 4),
        "Sign"       : sign,
        "Direction"  : direction
    })

weight_df = pd.DataFrame(weight_results).sort_values("Weight", ascending=False).reset_index(drop=True)

print("Descriptor Weights (fit on train set)\n")
print(weight_df.to_string(index=False))
print("\nTotal weight sum:", round(weight_df["Weight"].sum(), 4))

weights = weight_df.set_index("Descriptor")["Weight"]
signs   = weight_df.set_index("Descriptor")["Sign"]

Descriptor Weights (fit on train set)

      Descriptor   Delta  Weight  Sign    Direction
          fr_NH1  1.0824  1.0824     1 Glues_Higher
 NumHeterocycles  1.0300  1.0300     1 Glues_Higher
       RingCount  0.9247  0.9247     1 Glues_Higher
         NOCount  0.8410  0.8410     1 Glues_Higher
NumAromaticRings  0.8391  0.8391     1 Glues_Higher
      fr_aniline  0.6495  0.6495     1 Glues_Higher
    FractionCSP3 -0.5474  0.5474    -1  Glues_Lower

Total weight sum: 5.9141


RAW GLUESCORE (TRAIN SET)

In [ ]:
# Raw GlueScore on Train set (descriptive fit check, NOT validation)

weighted_z_train = z_train[desc_cols].multiply(weights * signs, axis=1)
train_df = train_df.copy()
train_df["GlueScore_raw"] = weighted_z_train.sum(axis=1)

print("Raw GlueScore for TRAIN SET (descriptive, fit check only)\n")
print("Glues mean:", round(train_df[train_df["Label"]==1]["GlueScore_raw"].mean(), 4))
print("Background mean:", round(train_df[train_df["Label"]==0]["GlueScore_raw"].mean(), 4))
print("\nGlues median:", round(train_df[train_df["Label"]==1]["GlueScore_raw"].median(), 4))
print("Background median:", round(train_df[train_df["Label"]==0]["GlueScore_raw"].median(), 4))

Raw GlueScore for TRAIN SET (descriptive, fit check only)

Glues mean: 2.9541
Background mean: -2.2662

Glues median: 2.7584
Background median: -2.6211


PERCENTILE CONVERSION (TRAIN SET)

In [ ]:
# Percentile Conversion (0-100 scale) on TRAIN set
train_df["GlueScore"] = train_df["GlueScore_raw"].rank(pct=True) * 100

print("GlueScore (0-100 Percentile Scale) for TRAIN SET\n")
print("Glues      — mean   :", round(train_df[train_df["Label"]==1]["GlueScore"].mean(), 1))
print("Background — mean   :", round(train_df[train_df["Label"]==0]["GlueScore"].mean(), 1))
print("\nGlues      — median :", round(train_df[train_df["Label"]==1]["GlueScore"].median(), 1))
print("Background — median :", round(train_df[train_df["Label"]==0]["GlueScore"].median(), 1))

GlueScore (0-100 Percentile Scale) for TRAIN SET

Glues      — mean   : 70.5
Background — mean   : 34.3

Glues      — median : 73.2
Background — median : 29.8


TRAIN SET AUC (FIT CHECK)

In [ ]:
# Train Set AUC (fit check, not validation)
# Will be compared against Step 6's test-set AUC to confirm no overfitting.
train_auc = roc_auc_score(train_df["Label"], train_df["GlueScore"])
print("\nTrain Set AUC (fit check, not validation):", round(train_auc, 3))


Train Set AUC (fit check, not validation): 0.862


SAVING OUTPUTS

In [ ]:
# Saving Step 5 outputs.

train_df.to_csv(DRIVE_DIR + "/Train_split.csv", index=False)
test_df.to_csv(DRIVE_DIR + "/Test_split.csv", index=False)

joblib.dump(scaler, DRIVE_DIR + "/Fitted_Scaler.pkl")

pd.Series(desc_cols, name="Descriptor").to_csv(DRIVE_DIR + "/Descriptor_list.csv", index=False)

weight_df.to_csv(DRIVE_DIR + "/GlueScore_Weight_Table.csv", index=False)

print("Step 5 Outputs Saved to Drive\n")
print("- Train_split.csv              : training molecules with GlueScore_raw and GlueScore added")
print("- Test_split.csv               : held-out test molecules (not yet scored)")
print("- Fitted_Scaler.pkl            : fitted StandardScaler (mean/std from train set, for standardizing new data)")
print("- Descriptor_list.csv          : List of the 7 final descriptor column names, in fixed order")
print("- GlueScore_Weight_Table.csv   : fitted GlueScore formula; Delta, Weight, Sign per descriptor")

Step 5 Outputs Saved to Drive

- Train_split.csv              : training molecules with GlueScore_raw and GlueScore added
- Test_split.csv               : held-out test molecules (not yet scored)
- Fitted_Scaler.pkl            : fitted StandardScaler (mean/std from train set, for standardizing new data)
- Descriptor_list.csv          : List of the 7 final descriptor column names, in fixed order
- GlueScore_Weight_Table.csv   : fitted GlueScore formula; Delta, Weight, Sign per descriptor
